# LaTeX export — figures and tables for the thesis document

Exports a figure and a table straight into the sibling thesis repository, so a chart in the written thesis is a regenerated artifact rather than a screenshot.

**Inputs:** joined train/test feature artifacts and their metadata contract (Stage 3 must have run)
**Outputs:** `../uas-master-thesis/figures/*.pdf` and `../uas-master-thesis/tables/*.tex` — **written outside this repository**

This notebook is not part of the `01` → `06` chain and has no `make` target; run it by hand while writing. The thesis preamble needs `\usepackage{booktabs}` for the exported table to compile.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Pins the same Stage-3 artifact paths and cohort constants every stage-4 notebook uses, so the exported figure describes exactly the data the models are trained and scored on. `THESIS_TEXTWIDTH_IN` is the thesis body text width (456.25555 pt): authoring the figure at that width means `width=\textwidth` scales it by 1.0 and the in-figure fonts land at their intended size.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from src.config import (
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    THESIS_TEXTWIDTH_IN,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.latex_export import save_figure, save_table

PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
WATER_LEVEL_COLUMN = f"{TARGET_STATION_ID}__water_level"

## Load the joined dataset

`load_joined_dataset()` is the repo's only sanctioned entry point into the Stage-3 artifacts: it validates the horizon, target station, and column contracts before returning anything, so a drifted contract fails here instead of silently producing a figure of the wrong thing.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
water_level = dataset.target_context_series[WATER_LEVEL_COLUMN]

## Figure — observed water level distribution

A histogram of every observed water level at the target station, authored at exactly the thesis text width. `save_figure` writes the PDF and prints the `figure` float; the figure stays open so it also renders inline here.

In [ ]:
fig, ax = plt.subplots(figsize=(THESIS_TEXTWIDTH_IN, THESIS_TEXTWIDTH_IN * 0.55))
ax.hist(water_level.dropna(), bins=60)
ax.set_xlabel("Water level [cm]")
ax.set_ylabel("Hourly observations")
ax.set_title(f"Observed water level at {TARGET_STATION_ID}")
ax.grid(alpha=0.25)
save_figure(
    fig,
    "target_water_level_hist",
    caption="Distribution of observed hourly water levels at the target station.",
)

## Table — summary statistics of the same series

`.describe()` of the plotted series, written as an `\input`-able booktabs fragment with no float wrapper — the wrapping `table` float is printed instead, ready to paste.

In [ ]:
summary = water_level.describe().to_frame()
display(summary)
save_table(
    summary,
    "target_water_level_describe",
    caption="Summary statistics of the observed water level at the target station.",
)

## Using the exports in the thesis

Each `save_*` call prints the float to paste into a chapter, with the path already relative to the thesis repository root — copy it as-is. The figure is authored at exactly `\textwidth`, so `width=\textwidth` scales it by 1.0 and its fonts match the body text.